In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path
import sys

pkg_path = str(Path(os.path.abspath('')).parent.absolute())
sys.path.insert(0, pkg_path)


data_path=pkg_path+'/results/'

from src import *

# Load config file
config = global_config.config
device = torch.device(config.device) 

In [ ]:

checkpoint_files = [f for f in os.listdir(data_path) if 'DTI_feat_' in f and f.endswith('.pth')]

# Initialize an empty list to store the data
data = []

# Iterate over each checkpoint file
for checkpoint_file in checkpoint_files:
    # Construct the full path to the checkpoint file
    checkpoint_path = os.path.join(data_path, checkpoint_file)
    
    # Load the checkpoint
    checkpoint = torch.load(checkpoint_path)
    
    # Extract the required values (train_loss and val_loss)
    epoch = checkpoint.get('epoch', None)  # Use 'latest' for the latest checkpoint
    loss = checkpoint.get('train_loss', None)
    val_loss = checkpoint.get('val_loss', None)

    if isinstance(train_loss, torch.Tensor):
        train_loss = train_loss.item()
    
    print(epoch)
    # Store the extracted information in the data list
    data.append({
        'epoch': epoch,
        'train_loss': loss,
        'val_loss': val_loss,
        'checkpoint_path': checkpoint_path
    })

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(data)

# Sort by epoch (if necessary)
df['epoch'] = pd.to_numeric(df['epoch'], errors='coerce')  # Ensure 'epoch' is numeric, convert 'latest' to NaN
df = df.sort_values(by='epoch')  # Sort by epoch number

# Display the resulting DataFrame
print(df)


In [ ]:
# Drop rows where val_loss is NaN
df = df.dropna(subset=['val_loss'])

# Sort by 'epoch' in ascending order
df = df.sort_values(by='epoch', ascending=True)
# Ensure 'train_loss' and 'val_loss' columns contain only the numerical values
df['train_loss'] = df['train_loss'].apply(lambda x: x.item() if isinstance(x, torch.Tensor) else x)

# Display the resulting DataFrame
print(df)

df.to_csv(data_path+'DTI_feat_losses.csv', index=False) 

In [ ]:
plt.figure(figsize=(10, 6))  # Set the figure size

# Plot training and validation losses
plt.plot(df['epoch'], df['train_loss'], label='Training Loss: weighted MSE', color='b', marker='o')
plt.plot(df['epoch'], df['val_loss'], label='Validation Loss: non-weighted MSE', color='g', marker='o')

# Add title and labels
plt.title('Loss over Epochs for DTI_AD_NC')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.ylim(0,1.5)

# Add legend
plt.legend()

# Show the plot
plt.grid(False)
plt.show()